# Drone or Bird? · Transfer learning on our own images

We adapt a pre-trained model to classify sky images as **drone** or **bird**.
We use the  `resnet18` model and replace its last layer with one for our 2 classes.

## Data

The folder names are the class names:

```text
content/dronecheck/
├── train/
│   ├── bird/    # 100 images
│   └── drone/   # 100 images
└── val/
    ├── bird/    # 25 images
    └── drone/   # 25 images
```

In [1]:
import torch
import torch.nn as nn
from torchvision import datasets, transforms, models

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Running on:", device)

Running on: cpu


## Helper functions

In [2]:
def evaluate(model, loader):
    # Turn off gradient tracking: we are only checking the model now.
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            labels = labels.to(device)
            predictions = model(images).argmax(1)
            correct += (predictions == labels).sum().item()
            total += len(labels)

    return correct / total


def train(model, train_loader, val_loader, epochs, learning_rate):
    model.to(device)

    # Only train the parameters that are not frozen.
    trainable_params = [
        param for param in model.parameters()
        if param.requires_grad
    ]
    optimizer = torch.optim.Adam(trainable_params, lr=learning_rate)
    loss_function = nn.CrossEntropyLoss()

    for epoch in range(1, epochs + 1):
        model.train()
        total_loss = 0

        for images, labels in train_loader:
            images = images.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()
            predictions = model(images)
            loss = loss_function(predictions, labels)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        val_accuracy = evaluate(model, val_loader)
        print(
            f"Epoch {epoch}/{epochs} | "
            f"loss: {total_loss:.3f} | "
            f"validation: {val_accuracy:.1%}"
        )

    return model

## Step 1 · Settings

In [3]:
DATA_PATH = "content/dronecheck"
ARCHITECTURE = "resnet18"
STRATEGY = "feature_extraction"   # or "fine_tuning"
EPOCHS = 8
IMAGE_SIZE = 224

## Step 2 · Check the data

In [4]:
train_data = datasets.ImageFolder(f"{DATA_PATH}/train")
val_data = datasets.ImageFolder(f"{DATA_PATH}/val")

CLASSES = train_data.classes
print("Classes:", CLASSES)
print("Training images:", len(train_data))
print("Validation images:", len(val_data))

Classes: ['bird', 'drone']
Training images: 200
Validation images: 50


## Step 3 · Build the model

In [5]:
weights = models.get_model_weights(ARCHITECTURE).DEFAULT
model = models.get_model(ARCHITECTURE, weights=weights)

# feature_extraction: freeze the pre-trained layers, train only the new last layer
for param in model.parameters():
    param.requires_grad = (STRATEGY == "fine_tuning")

if hasattr(model, "fc"):   # ResNet
    model.fc = nn.Linear(model.fc.in_features, len(CLASSES))
else:                      # MobileNet, EfficientNet
    model.classifier[-1] = nn.Linear(model.classifier[-1].in_features, len(CLASSES))

## Step 4 · Prepare the images

In [6]:
normalize = transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                 std=[0.229, 0.224, 0.225])

train_transform = transforms.Compose([
    transforms.RandomResizedCrop(IMAGE_SIZE, scale=(0.7, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    normalize,
])
val_transform = transforms.Compose([
    transforms.Resize(IMAGE_SIZE),
    transforms.ToTensor(),
    normalize,
])

train_dataset = datasets.ImageFolder(f"{DATA_PATH}/train", transform=train_transform)
val_dataset = datasets.ImageFolder(f"{DATA_PATH}/val", transform=val_transform)

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = torch.utils.data.DataLoader(val_dataset, batch_size=32)

## Step 5 · Train and evaluate

In [7]:
learning_rate = 1e-3 if STRATEGY == "feature_extraction" else 1e-4

model = train(model, train_loader, val_loader, epochs=EPOCHS, learning_rate=learning_rate)

print(f"Final validation accuracy: {evaluate(model, val_loader):.1%}")

Epoch 1/8 | loss: 8.717 | validation: 78.0%
Epoch 2/8 | loss: 7.168 | validation: 68.0%
Epoch 3/8 | loss: 5.370 | validation: 82.0%
Epoch 4/8 | loss: 4.352 | validation: 72.0%
Epoch 5/8 | loss: 4.398 | validation: 84.0%
Epoch 6/8 | loss: 3.675 | validation: 82.0%
Epoch 7/8 | loss: 3.663 | validation: 86.0%
Epoch 8/8 | loss: 3.817 | validation: 86.0%
Final validation accuracy: 86.0%


## Step 6 · Save model

In [8]:
# import os
# os.makedirs("../model", exist_ok=True)
# torch.save({
#     "architecture": ARCHITECTURE,
#     "classes": CLASSES,
#     "image_size": IMAGE_SIZE,
#     "state_dict": model.state_dict(),
# }, "../model/drone_check.pth")
# print("Saved ../model/drone_check.pth")

Saved ../model/drone_check.pth
